In [1]:
import sys
import os

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import datasets
from functools import partial

import torch

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
model_path = "../../self-corrective-llama_untrained"
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

model_config = AutoConfig.from_pretrained(model_path)
model_config.alpha_boost = 5.0
model_config.tau = 0.7
model_config.max_boost = 8.0

model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True,)

print(tokenizer.encode("<DEL_S>"))
print(tokenizer.encode("<DEL_A>"))

[128000, 128256]
[128000, 128257]


In [3]:
dataset = datasets.load_from_disk("../../dataset/training")
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 31519
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 3503
})


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
# sample = train_dataset[0]
# sample["input_ids"][391] = 128256
# sample["input_ids"][392] = 128257

# sample["labels"][391] = 128256
# sample["labels"][392] = 128257

# sample["hallucination_labels"][391] = 1
# sample["hallucination_labels"][392] = 2

# sample["input_ids"] = torch.tensor([sample["input_ids"][390:420]])
# sample["attention_mask"] = torch.tensor([sample["attention_mask"][390:420]])
# sample["labels"] = torch.tensor([sample["labels"][390:420]])
# sample["hallucination_labels"] = torch.tensor([sample["hallucination_labels"][390:420]])


sample = train_dataset[2]
sample["input_ids"] = torch.tensor([sample["input_ids"][290:]])
sample["attention_mask"] = torch.tensor([sample["attention_mask"][290:]])
sample["labels"] = torch.tensor([sample["labels"][290:]])
sample["hallucination_labels"] = torch.tensor([sample["hallucination_labels"][290:]])

print(sample)

{'input_ids': tensor([[128007,    791,   1920,    315,  18475,    374,   6485,     11,    439,
            433,  14117,    389,  12387,  45964,  15207,   3956,    430,  11951,
          16681,    311,   2804,    279,   1920,     13, 128256,    420,   6485,
           1920, 128009]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]]), 'labels': tensor([[  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100, 128256,    420,   6485,
           1920, 128009]]), 'hallucination_labels': tensor([[-100,    0,    0,    0,    0,    0,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    0,    0,    0,    0]])}


In [15]:
model.train()
model.forward(
    input_ids=sample["input_ids"], 
    attention_mask=sample["attention_mask"], 
    labels=sample["labels"], 
    hallucination_labels=sample["hallucination_labels"]
)

Hallucination probs:
tensor([[[0.4703],
         [0.6257],
         [0.4389],
         [0.5686],
         [0.4234],
         [0.5252],
         [0.5958],
         [0.5441],
         [0.4754],
         [0.5312],
         [0.5307],
         [0.6435],
         [0.6842],
         [0.5546],
         [0.7180],
         [0.5651],
         [0.5945],
         [0.6114],
         [0.4217],
         [0.5162],
         [0.4778],
         [0.4897],
         [0.5401],
         [0.7688],
         [0.7273],
         [0.7359],
         [0.6642],
         [0.5919],
         [0.3311]]], grad_fn=<SigmoidBackward0>)
Training
Mask S label:
tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False,  True, False, False, False, False]])
Mask A label:
tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False,

SelfCorrectiveLlamaOutput(loss=None, logits=tensor([[[ 3.9723e+00,  4.9270e+00,  6.5004e+00,  ..., -2.2485e+00,
          -4.7890e-03, -4.7890e-03],
         [ 3.5925e+00,  6.2215e+00,  6.3131e+00,  ..., -4.1484e+00,
          -5.8431e-03, -5.8431e-03],
         [ 1.0203e+01,  1.0968e+01,  7.8770e+00,  ..., -1.6350e+00,
          -3.7182e-03, -3.7182e-03],
         ...,
         [ 7.5493e+00,  9.0993e+00,  6.6013e+00,  ..., -1.0823e+00,
          -6.3768e-03, -6.3768e-03],
         [ 9.7720e+00,  9.9746e+00,  8.1995e+00,  ..., -9.4697e-01,
          -5.1076e-03, -5.1076e-03],
         [-4.1096e-01,  7.3959e-01,  2.8673e+00,  ..., -1.6576e+00,
          -1.2853e-03, -1.2853e-03]]], grad_fn=<CatBackward0>), past_key_values=DynamicCache(layers=[<transformers.cache_utils.DynamicLayer object at 0x106f2d7d0>, <transformers.cache_utils.DynamicLayer object at 0x131d40e90>, <transformers.cache_utils.DynamicLayer object at 0x131d40550>, <transformers.cache_utils.DynamicLayer object at 0x32b1a919

In [ ]:
model.eval()

# text = "What is the capital of France?"
sample = train_dataset[0]
input_ids = tokenizer.encode(text)
input_ids = torch.tensor([input_ids])
attention_mask = torch.ones_like(input_ids)

# input_ids = torch.tensor([sample["input_ids"]])
# attention_mask = torch.ones_like(torch.tensor([sample["attention_mask"]]))

result = model.generate(input_ids=input_ids, attention_mask=attention_mask)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Hallucination probs:
tensor([[[0.5470],
         [0.6649],
         [0.6157],
         [0.6344],
         [0.6873],
         [0.6125],
         [0.5198],
         [0.6262]]])
Inference
Gate mask:
tensor([[[False],
         [ True],
         [ True],
         [ True],
         [ True],
         [ True],
         [False],
         [ True]]])
Hallucination probs:
tensor([[[0.5470],
         [0.6649],
         [0.6157],
         [0.6344],
         [0.6873],
         [0.6125],
         [0.5198],
         [0.6262]]])
new logits:
tensor([[[-0.0050, -0.0050],
         [-0.0086, -0.0086],
         [-0.0060, -0.0060],
         [-0.0099, -0.0099],
         [-0.0123, -0.0123],
         [-0.0077, -0.0077],
         [-0.0119, -0.0119],
         [-0.0037, -0.0037]]])
Boosted logits:
tensor([[[0.5420, 0.5420],
         [0.6563, 0.6563],
         [0.6097, 0.6097],
         [0.6246, 0.6246],
         [0.6750, 0.6750],
         [0.6048, 0.6048],
         [0.5079, 0.5079],
         [0.6225, 0.6225]]])
Gat

In [13]:
res = tokenizer.decode(result[0])
print(res)

<|begin_of_text|>What is the capital of France? Paris.
The capital of France is Paris. Paris is the most populous city in France and is known
